# Saved-Artifact Analysis Run

This notebook wraps the analysis scripts that summarize saved validation and method-specific motorneuron graph artifacts. Run it after the c-GC, c-GC*, LPCMCI, and OASIS motorneuron notebooks have produced their outputs. No legacy combined or hindbrain notebook output is required.

By default the commands are only printed. Set `RUN_ANALYSIS = True` to execute the selected steps.

<!-- reviewer-resume-contract -->
## Execution and resume contract

This notebook is aligned with the reviewer-revision implementation. Expensive work is checkpointed and safe to restart with the same configuration. Do not change methods, seeds, thresholds, or output paths while resuming. Saved outputs remain provisional until the compute-machine run and verification gates complete.


In [ ]:
from pathlib import Path
import json
import os
import shlex
import subprocess
import sys


def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd().resolve() if start is None else start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "calcium_transient_rising_flank").is_dir():
            return candidate
        nested = candidate / "calcium-transient-rising-flank"
        if (nested / "src" / "calcium_transient_rising_flank").is_dir():
            return nested
    raise RuntimeError("Run from the repository, package root, or notebooks directory.")


PROJECT_ROOT = find_project_root()
PYTHON = sys.executable
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from calcium_transient_rising_flank.checkpointing import format_progress
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
def command_text(command: list[str]) -> str:
    return " ".join(shlex.quote(str(part)) for part in command)


def _valid_summary(command: list[str]) -> bool:
    if '--output-dir' not in command:
        return False
    marker = Path(command[command.index('--output-dir') + 1]) / 'summary.json'
    if not marker.is_file():
        return False
    try:
        payload = json.loads(marker.read_text())
        return isinstance(payload, dict) and payload.get('status') == 'complete'
    except (OSError, json.JSONDecodeError):
        return False


def run_steps(steps: list[tuple[str, list[str]]], *, execute: bool, resume: bool = True) -> None:
    env = os.environ.copy()
    env["PYTHONPATH"] = str(PROJECT_ROOT / "src")
    env["PYTHONUNBUFFERED"] = "1"
    env.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")
    env.setdefault("XDG_CACHE_HOME", "/tmp/font-cache")
    total = len(steps)
    completed = 0
    if total:
        print(format_progress(completed, total, label='Analysis stages'), flush=True)
    for index, (name, command) in enumerate(steps, start=1):
        print(f"[stage] {index}/{total}: {name}", flush=True)
        print(command_text(command), flush=True)
        if resume and _valid_summary(command):
            completed += 1
            print('   resumed: valid summary.json already exists', flush=True)
            print(format_progress(completed, total, label='Analysis stages') + ' | loaded completed output', flush=True)
            continue
        if execute:
            subprocess.run(command, cwd=PROJECT_ROOT, env=env, check=True)
            completed += 1
            print(format_progress(completed, total, label='Analysis stages') + f' | completed {name}', flush=True)
    if not execute:
        print("Dry run only. Set RUN_ANALYSIS = True to execute.")


In [ ]:
RUN_ANALYSIS = False
RESUME_ANALYSIS = False  # rebuild cheap analyses after method outputs change

RUN_CHEN_COMPARISON = True
RUN_EMPIRICAL_PAIRING = True
RUN_GRAPH_STABILITY = True
RUN_FALSE_POSITIVE_TRADEOFFS = True
RUN_READINESS_REPORT = True
RUN_EVIDENCE_PACKAGE = True
RUN_TODO_AUDIT = True

OUTPUT_ROOT = PROJECT_ROOT / "outputs"
MOTORNEURON_OUTPUT_DIR = OUTPUT_ROOT / "motorneurons"
LPCMCI_OUTPUT_DIR = OUTPUT_ROOT / "revision_campaign" / "motorneurons_lpcmci"
OASIS_OUTPUT_DIR = OUTPUT_ROOT / "revision_campaign" / "motorneurons_oasis"
VALIDATION_RESULTS_DIR = OUTPUT_ROOT / "validation_results"

In [ ]:
steps = []
if RUN_CHEN_COMPARISON:
    steps.append(
        (
            "Chen and saved graph comparison",
            [PYTHON, "examples/build_chen_comparison_table.py", "--input-dir", str(MOTORNEURON_OUTPUT_DIR), "--lpcmci-input-dir", str(LPCMCI_OUTPUT_DIR), "--oasis-input-dir", str(OASIS_OUTPUT_DIR), "--output-dir", str(OUTPUT_ROOT / "chen_comparison")],
        )
    )
if RUN_EMPIRICAL_PAIRING:
    steps.append(
        (
            "paired rise-minus-fall empirical tests",
            [PYTHON, "examples/analyze_empirical_pairing.py", "--input", str(OUTPUT_ROOT / "chen_comparison" / "chen_comparison_rows.csv"), "--output-dir", str(OUTPUT_ROOT / "empirical_stats")],
        )
    )
if RUN_GRAPH_STABILITY:
    steps.append(
        (
            "saved graph support and stability",
            [PYTHON, "examples/analyze_graph_stability.py", "--input-dir", str(MOTORNEURON_OUTPUT_DIR), "--lpcmci-input-dir", str(LPCMCI_OUTPUT_DIR), "--oasis-input-dir", str(OASIS_OUTPUT_DIR), "--output-dir", str(OUTPUT_ROOT / "graph_stability")],
        )
    )
if RUN_FALSE_POSITIVE_TRADEOFFS:
    steps.append(
        (
            "synthetic recall and false-positive tradeoffs",
            [PYTHON, "examples/analyze_false_positive_tradeoffs.py", "--input-dir", str(VALIDATION_RESULTS_DIR), "--output-dir", str(OUTPUT_ROOT / "validation_tradeoffs")],
        )
    )
if RUN_READINESS_REPORT:
    steps.append(
        (
            "result readiness report",
            [PYTHON, "examples/build_result_readiness_report.py", "--output-root", str(OUTPUT_ROOT), "--output-dir", str(OUTPUT_ROOT / "result_readiness")],
        )
    )
if RUN_EVIDENCE_PACKAGE:
    steps.append(
        (
            "manuscript evidence package",
            [PYTHON, "examples/build_manuscript_evidence_package.py", "--output-root", str(OUTPUT_ROOT), "--output-dir", str(OUTPUT_ROOT / "manuscript_evidence")],
        )
    )
if RUN_TODO_AUDIT:
    steps.append(
        (
            "todo completion audit",
            [PYTHON, "examples/build_todo_completion_audit.py", "--output-dir", str(OUTPUT_ROOT / "todo_completion")],
        )
    )

run_steps(steps, execute=RUN_ANALYSIS, resume=RESUME_ANALYSIS)


In [ ]:
readiness_path = OUTPUT_ROOT / "result_readiness" / "summary.json"
if readiness_path.is_file():
    print(json.dumps(json.loads(readiness_path.read_text()), indent=2))
else:
    print(f"No readiness summary yet at {readiness_path}")